# TV3 — Logistic Regression (scikit-learn, KHÔNG dùng Spark)

Chạy các cell từ trên xuống. Dùng nguyên `features_v1` của TV2; giữ train/val/test TV1. **Không đọc test trong notebook này.**

In [ ]:
# CELL 1 — Cài thư viện (không cài Spark)
%pip install -q "numpy>=1.26,<3" "pandas>=2.1,<3" "pyarrow>=14,<24" "scikit-learn>=1.4,<2" "matplotlib>=3.7,<4" "joblib>=1.3,<2"


In [ ]:
# CELL 2 — Kết nối Drive và chọn đường dẫn (SỬA các biến sau cho phù hợp)
import sys, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Repo: thư mục chứa src/model_lr.py, src/model_rf.py, notebooks/... đã tải từ GitHub
REPO = Path('/content/BDA-Nhom_8-main')
# Drive của TV2: thư mục có train/, val/, test/ và manifest.json
FEATURES_ON_DRIVE = Path('/content/drive/MyDrive/BigData_Topic20/01_data_frozen/features_v1')
# Kết quả lưu trên Drive, tránh mất dữ liệu khi Colab ngắt phiên
OUTPUT_ROOT = Path('/content/drive/MyDrive/BigData_Topic20/02_models/tv3_sklearn')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / 'src' / 'tv3_common.py').is_file():
    raise FileNotFoundError(f'Thiếu code TV3 tại {REPO}. Hãy upload/clone repo đã ghép file TV3.')
if not (FEATURES_ON_DRIVE / 'train').is_dir():
    raise FileNotFoundError(f'Chưa thấy bảng feature TV2: {FEATURES_ON_DRIVE}')

# Copy dữ liệu sang đĩa cục bộ Colab cho tốc độ đọc tốt hơn.
FEATURES_DIR = Path('/content/features_v1')
if not (FEATURES_DIR / 'train').is_dir():
    shutil.copytree(FEATURES_ON_DRIVE, FEATURES_DIR, dirs_exist_ok=True)
print('REPO:', REPO)
print('FEATURES_DIR:', FEATURES_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
sys.path.insert(0, str(REPO))


In [ ]:
# CELL 3 — Đọc dữ liệu cố định và kiểm tra số dòng/schema
from src.tv3_common import read_split, CAT_COLS, NUM_COLS, FEATURE_COLS, assert_disjoint
train = read_split(FEATURES_DIR, 'train')
val = read_split(FEATURES_DIR, 'val')
assert_disjoint(train, val)
print('Train:', train.shape, 'Validation:', val.shape)
print('Biến phân loại:', CAT_COLS)
print('Biến số:', NUM_COLS)
print('Tỷ lệ trễ Train:', round(train['Delay'].mean(), 4))
display(train.head(5))


## CELL 4 — Baseline 1 (lớp đa số) và baseline 2 (trễ theo hãng)

In [ ]:
from src.tv3_common import evaluate_baselines
baselines = evaluate_baselines(train, val)
display(baselines[['baseline', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'threshold']])

## CELL 5 — Huấn luyện Logistic Regression, quét C × l1_ratio

Quy tắc: Fit tiền xử lý chỉ trên Train, chọn cấu hình có ROC-AUC tốt nhất trên Validation, chọn threshold tối đa F1 trên Validation.

In [ ]:
from src.model_lr import train_lr
lr_metrics = train_lr(FEATURES_DIR, OUTPUT_ROOT)
print('Validation metrics LR:')
for name, val in lr_metrics.items():
    print(name, ':', val)

## CELL 6 — Bảng thử nghiệm, các hệ số LR và biểu đồ

In [ ]:
import pandas as pd
from IPython.display import display, Image
res = OUTPUT_ROOT / 'results'
exp = pd.read_csv(res / 'lr_experiments.csv')
display(exp.sort_values(['roc_auc', 'pr_auc'], ascending=False))
display(pd.read_csv(res / 'lr_coefficients.csv').head(20))
for plot in ['lr_roc.png', 'lr_pr.png', 'lr_confusion.png']:
    display(Image(filename=str(OUTPUT_ROOT / 'reports' / 'tv3' / plot)))

## Hoàn thành LR
Đã lưu `models/lr/best_model.joblib`, `models/lr/config.json`, dự đoán validation và biểu đồ trong `OUTPUT_ROOT`. **Không dùng tập Test** ở notebook này. Sau khi huấn luyện RF và chốt cả hai mô hình, dùng notebook `06_test_tv3.ipynb`.